# MCP-Powered AI Assistant
### A practical Model Context Protocol (MCP) client with tool discovery and optional LLM integration

This starter notebook demonstrates how an AI application can discover and call tools exposed by an MCP server:
1. Create a local MCP server with three tools.
2. Connect to it using an MCP client.
3. Discover tools dynamically and call them with structured arguments.
4. Optionally let an OpenAI-compatible LLM select a tool.
5. Use a deterministic fallback even without API credentials.

Keep API keys in environment variables or a `.env` file; never hard-code secrets in notebook cells.

## 1. Install dependencies
Run in a fresh Python environment. Restart the kernel if your notebook environment requests it.

In [ ]:
%pip install -q "mcp[cli]>=1.9.0" "openai>=1.68.0" python-dotenv

## 2. Load optional LLM configuration
Create a `.env` file beside the notebook if you want to enable LLM-assisted tool selection. The MCP tool demo works without credentials.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
print("LLM credentials configured:", bool(API_KEY))
print("Configured model:", MODEL)

## 3. Write the MCP server
This cell creates `mcp_server.py`. The server offers an arithmetic tool, a tiny keyword-search tool, and a status tool.

In [ ]:
server_code = r'''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("portfolio-assistant")
KNOWLEDGE_BASE = [
    {"title": "MCP overview", "text": "Model Context Protocol connects AI applications to tools and context sources."},
    {"title": "Tool discovery", "text": "MCP clients can discover tools and call them using structured arguments."},
    {"title": "Security", "text": "Use least-privilege tools, validate inputs, protect secrets, and require confirmation for consequential actions."},
]

@mcp.tool()
def add_numbers(a: float, b: float) -> dict:
    """Add two numbers and return the result."""
    return {"a": a, "b": b, "sum": a + b}

@mcp.tool()
def search_knowledge_base(query: str, limit: int = 3) -> dict:
    """Search a tiny demo knowledge base using keyword matching."""
    terms = [term.lower() for term in query.split() if term.strip()]
    scored = []
    for doc in KNOWLEDGE_BASE:
        haystack = (doc["title"] + " " + doc["text"]).lower()
        score = sum(haystack.count(term) for term in terms)
        if score:
            scored.append({"title": doc["title"], "text": doc["text"], "score": score})
    scored.sort(key=lambda item: item["score"], reverse=True)
    return {"query": query, "results": scored[:max(1, min(limit, 10))]}

@mcp.tool()
def get_system_status() -> dict:
    """Return a harmless demo status response."""
    return {"status": "ok", "service": "portfolio-assistant", "environment": "local-demo"}

if __name__ == "__main__":
    mcp.run()
'''
with open("mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)
print("Created mcp_server.py")

## 4. Discover tools through the MCP client
The client launches the local server as a child process over stdio. Run this notebook from the folder where `mcp_server.py` was created.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def list_mcp_tools(server_script="mcp_server.py"):
    params = StdioServerParameters(command="python", args=[server_script], env=os.environ.copy())
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return [
                {"name": tool.name, "description": tool.description, "inputSchema": tool.inputSchema}
                for tool in result.tools
            ]

discovered_tools = await list_mcp_tools()
for tool in discovered_tools:
    print(f"- {tool['name']}: {tool['description']}")

## 5. Call a tool directly
Test the MCP connection without using an LLM.

In [ ]:
async def call_mcp_tool(tool_name, arguments, server_script="mcp_server.py"):
    params = StdioServerParameters(command="python", args=[server_script], env=os.environ.copy())
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments=arguments)
            output = []
            for item in result.content:
                if getattr(item, "text", None):
                    output.append(item.text)
                else:
                    output.append(str(item))
            return {"isError": result.isError, "content": output}

print(await call_mcp_tool("add_numbers", {"a": 17, "b": 25}))
print(await call_mcp_tool("search_knowledge_base", {"query": "MCP tools", "limit": 3}))

## 6. Optional LLM-assisted tool selection
For readability, this starter asks the model for a JSON tool choice. In production, prefer your provider's native tool-calling API and validate every argument against the discovered JSON schema.

In [ ]:
import json
import re

async def fallback_route(user_request):
    text = user_request.lower()
    if "add" in text or "sum" in text:
        nums = re.findall(r"-?\\d+(?:\\.\\d+)?", user_request)
        if len(nums) >= 2:
            return await call_mcp_tool("add_numbers", {"a": float(nums[0]), "b": float(nums[1])})
    if "search" in text or "mcp" in text or "knowledge" in text:
        return await call_mcp_tool("search_knowledge_base", {"query": user_request, "limit": 3})
    if "status" in text:
        return await call_mcp_tool("get_system_status", {})
    return {"message": "Try 'add 17 and 25', 'search MCP tools', or 'get status'."}

async def ask_assistant(user_request: str):
    if not API_KEY:
        return {
            "mode": "deterministic-fallback",
            "message": "Set OPENAI_API_KEY to enable LLM planning.",
            "tool_result": await fallback_route(user_request),
        }

    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)
    prompt = (
        "Choose one suitable tool and return ONLY valid JSON with keys tool_name and arguments. "
        "If none fits, use tool_name=none and empty arguments. Available tools: "
        + json.dumps(discovered_tools)
        + "\\nUser request: " + user_request
    )
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Choose only a listed tool. Output valid JSON, no markdown."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    decision = json.loads(response.choices[0].message.content or "{}")
    tool_name = decision.get("tool_name")
    arguments = decision.get("arguments", {})
    allowed = {tool["name"] for tool in discovered_tools}
    if tool_name not in allowed:
        return {"mode": "llm", "message": "No suitable tool selected.", "decision": decision}
    if not isinstance(arguments, dict):
        raise ValueError("Tool arguments must be a JSON object")
    return {
        "mode": "llm",
        "tool": tool_name,
        "arguments": arguments,
        "tool_result": await call_mcp_tool(tool_name, arguments),
    }

await ask_assistant("Add 17 and 25")

## 7. Try example requests

In [ ]:
for request in [
    "Add 12 and 30",
    "Search the knowledge base for MCP security",
    "Get system status",
]:
    print("\\nUSER:", request)
    print(await ask_assistant(request))

## 8. Architecture and safety notes

- **MCP server:** exposes tools through the Model Context Protocol.
- **MCP client:** discovers and invokes those tools using structured arguments.
- **LLM planner:** optionally maps natural-language requests to a tool and arguments.
- **Security:** this is a local educational demo. Production systems should validate every argument against the tool schema, enforce authorization, use timeouts and audit logs, and require confirmation for consequential actions.
- **Secrets:** store credentials in environment variables or a secrets manager; do not commit `.env`.
- **Limitations:** the sample knowledge search is simple keyword matching, not a vector database or production search system.

## 9. Suggested extensions

1. Add a web-search MCP tool using an approved API.
2. Add a read-only database MCP tool.
3. Build a Streamlit or React chat interface.
4. Add authentication, permissions, retries, and tests.
5. Add tracing for tool-call latency, failures, and model cost.